In [1]:
from src.egnn_tda.dataset import *
from src.egnn_tda.models import *
from src.egnn_tda.train import *

import src.templates
import plotly.io as pio
import plotly.graph_objects as go
pio.templates.default = "template_TNR"

C:\Users\alwas\Desktop\egnn_tda\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = QM9Dataset(root="./dataset/qm9_v1.3_total",
                    transform_list = None,
                    pre_transform_list = [TDA_transform(), RDKitAromaticSPPreTransform(), Add_node_attrs(), DropFields("smiles", "name", "z", "x")],
                    pre_filter_list = [MaxAtomsFilter(max_atoms= 1000)],
                    force_reload = False
)

Processing...
  0%|          | 0/133885 [00:00<?, ?it/s]C:\Users\alwas\Desktop\egnn_tda\.venv\Lib\site-packages\ripser\ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
C:\Users\alwas\Desktop\egnn_tda\.venv\Lib\site-packages\ripser\ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
100%|██████████| 133885/133885 [04:53<00:00, 456.32it/s]
Done!


In [3]:
model = EGNN(node_attr_dim = dataset.node_attr.shape[1],
              edge_attr_dim   = dataset.edge_attr.shape[1],
              hidden_dim = 64,
              num_layers = 7,
              equivariant=False
              )

In [4]:
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import torch

n = len(dataset)
n_train = int(0.90 * n)
n_test = n - n_train
train_ds, test_ds = random_split(dataset, [n_train, n_test], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

In [5]:
n_epochs = 500
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-16)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

train_losses, val_losses = [], []

In [6]:
model.to(device)

_train_losses, _val_losses, y_mean = train( model = model,
                                            train_loader = train_loader,
                                            val_loader = test_loader,
                                            optimizer = optimizer,
                                            scheduler = scheduler,
                                            epochs = n_epochs,
                                            device = device,
                                            print_every_epoch=1000,
                                            ckpt_path="model_gnn_v1.3_total.pt",
                                            )
train_losses += _train_losses
val_losses += _val_losses

Epoch: 100%|██████████| 500/500 [7:07:14<00:00, 51.27s/it, lr=0.00e+00, train_mse=1.5315e-03, val_mse=2.7482e-02]  


In [7]:
torch.save({"model_state": model.state_dict()}, "model_gnn_v1.3_total_overfitted.pt")

In [11]:
ckpt = torch.load("model_gnn_v1.3_total_overfitted.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state"])
#optimizer.load_state_dict(ckpt["optimizer_state"])
#scheduler.load_state_dict(ckpt["scheduler_state"])

<All keys matched successfully>

In [9]:
fig = go.Figure(layout={
        'plot_bgcolor': 'white',
        'paper_bgcolor' : 'white',})

fig.update_layout(width = 600,
                  height = 600,
                  legend = dict(x = 0.95, y = 0.9)
                  )

fig.update_xaxes(title = "epoch")
fig.update_yaxes(title = "MSE loss")

fig.add_trace(go.Scatter(y=val_losses, mode='lines', name = 'Validation loss'))
fig.add_trace(go.Scatter(y=train_losses, mode='lines', name = 'Training loss'))